In [1]:
import pandas as pd
from pickle import dump

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

In [2]:
df_path = 'data/csv_files/new_vk_areas_df.csv'
df_with_pahs_path = 'data/csv_files/new_vk_areas_with_pahs.csv'

weighted = True

In [3]:
df = pd.read_csv(df_path)
df_with_pahs = pd.read_csv(df_with_pahs_path)

In [4]:
columns_3d = ['O/C','H/C', 'N/C']
X_3d = df[columns_3d].to_numpy()
X_3d_scaled = StandardScaler().fit_transform(X_3d)

y = df['category'].to_numpy()
weights = df['weights'].to_numpy() if weighted else None

X_3d_with_pahs = df_with_pahs[columns_3d].to_numpy()
X_3d_with_pahs_scaled = StandardScaler().fit_transform(X_3d_with_pahs)

y_with_pahs = df_with_pahs['category'].to_numpy()

weights_with_pahs = df_with_pahs['weights'].to_numpy() if weighted else None

In [5]:
def find_best_hyperparams(clf,params_dictionary,X,y,sample_weight=None,refit=True, verbose=3, scoring='balanced_accuracy',
                          n_jobs=-1):
    grid = GridSearchCV(clf,params_dictionary, refit=refit, verbose=verbose, scoring=scoring, n_jobs=n_jobs)
    grid.fit(**dict(X=X,y=y,sample_weight=sample_weight) if sample_weight is not None else dict(X=X,y=y))
    return grid

## Optimise KNN

In [ ]:
param_grid_knn = {
    'weights':['distance'],
    'n_neighbors': [1,5,10,50,100,200],
    }

In [ ]:
grid_knn_3d = find_best_hyperparams(KNeighborsClassifier(),param_grid_knn,X_3d_scaled,y)
print('3D KNN best_estimator_:', grid_knn_3d.best_estimator_)

In [ ]:
grid_knn_pahs_3d = find_best_hyperparams(KNeighborsClassifier(),param_grid_knn,X_3d_with_pahs_scaled,y_with_pahs)
print('3D KNN with PAHs best_estimator_:', grid_knn_pahs_3d.best_estimator_)

## Optimising RBF SVM
The usual `GridSearchCV` doesn't seem to work properly for RBF: it outputs values of 100,000 for C, which is patently nonsensical for a dataset of <10,000 observations. Therefore, another – unfortunately slower – approach is taken.

In [ ]:
param_grid_svm_rbf = {
    'kernel':['rbf'],
    'C': [10**(x) for x in range(7)],
    'gamma': ['auto'] + [10**(-x) for x in range(7)],
    }
param_grid_svm_rbf

In [ ]:
grid_rbf_svm_3d=find_best_hyperparams(SVC(),param_grid_svm_rbf,X_3d_scaled,y,sample_weight=weights)
print('3D RBF SVM best_estimator_:', grid_rbf_svm_3d.best_estimator_)

In [ ]:
# print('3D SVM (RBF) with PAHs best_estimator_', rbf_opt(param_grid_svm_rbf,X_3d_with_pahs,y_with_pahs))
grid_rbf_svm_pahs_3d = find_best_hyperparams(SVC(),param_grid_svm_rbf,X_3d_with_pahs_scaled,y_with_pahs,sample_weight=weights_with_pahs)
print('3D RBF SVM with PAHs best_estimator_:', grid_rbf_svm_pahs_3d.best_estimator_)

## Optimising Polynomial SVM

In [ ]:
# choose C and gamma hyperparameters for SVM with polynomial kernel for the 3D case
# https://www.geeksforgeeks.org/machine-learning/svm-hyperparameter-tuning-using-gridsearchcv-ml/

param_grid_poly_svm = {
    'kernel':['poly'],
    'degree': list(range(1,4)),
    'C': [10**x for x in range(6)],
    'gamma': ['auto'] + [10**(-x) for x in range(6)],
    }
param_grid_poly_svm

In [ ]:
grid_poly_svm_3d = find_best_hyperparams(SVC(),param_grid_poly_svm,X_3d_scaled,y,sample_weight=weights)
print('3D poly SVM best_estimator_:', grid_poly_svm_3d.best_estimator_)

In [ ]:
grid_poly_svm_pahs_3d = find_best_hyperparams(SVC(),param_grid_poly_svm,X_3d_with_pahs_scaled,y_with_pahs,sample_weight=weights_with_pahs)
print('3D poly SVM with PAHs best_estimator_:', grid_poly_svm_pahs_3d.best_estimator_)

In [ ]:
text_file = open('data/hyperparameters/hyperparameters.txt', 'w')

text_file.write(f'''3D KNN best_estimator_: {grid_knn_3d.best_estimator_}
3D KNN with PAHs best_estimator_: {grid_knn_pahs_3d.best_estimator_}

3D RBF SVM best_estimator_: {grid_rbf_svm_3d.best_estimator_}
3D RBF SVM with PAHs best_estimator_: {grid_rbf_svm_pahs_3d.best_estimator_}
                
3D SVM (poly) best_estimator_: {grid_poly_svm_3d.best_estimator_}
3D SVM (poly) with PAHs best_estimator_: {grid_poly_svm_pahs_3d.best_estimator_}
''')

text_file.close()

In [ ]:
hyperparameters = {
    'knn_3d':grid_knn_3d.best_params_,
    'knn_pahs_3d':grid_knn_pahs_3d.best_params_,
    'rbf_svm_3d':grid_rbf_svm_3d.best_params_,
    'rbf_svm_pahs_3d':grid_rbf_svm_pahs_3d.best_params_,
    'poly_svm_3d':grid_poly_svm_3d.best_params_,
    'poly_svm_pahs_3d':grid_poly_svm_pahs_3d.best_params_
}
with open(f'data/hyperparameters/hyperparameters.pkl', 'wb') as f:
    dump(hyperparameters, f)